# Implementing the RAG Flow (Lesson 36)

Follow-along notebook for *Implementing the RAG flow* in Building with the Claude API.

The lesson uses several "pre-built" pieces you don't have. They're all defined here:
- `chunk_by_section` (from the chunking lesson)
- `generate_embedding`, updated to accept a string **or** a list of strings
- `VectorIndex`, a small in-memory vector store with `add_vector` and `search`
- a sample `report.md` shaped like the lesson's (table of contents, methodology, numbered sections, and an incident ID the BM25 lesson uses next)

Then it runs the lesson's five steps, plus an optional sixth: handing the retrieved chunks to Claude to answer the question.

**Needs:** `ANTHROPIC_API_KEY` and `VOYAGE_API_KEY` in `.env`. **Run order:** Run All after a kernel restart.

## 1. Setup

In [1]:
import re
from math import sqrt
from pathlib import Path

import voyageai
from anthropic import Anthropic
from dotenv import load_dotenv
import os
from voyageai import Client

load_dotenv()
voyage_client = Client(api_key=os.getenv("VOYAGE_API_KEY"), max_retries=3)
anthropic_client = Anthropic()

EMBED_MODEL = "voyage-3-large"
MODEL = "claude-haiku-4-5"

c:\Users\bspedden\OneDrive - The Select Group LLC\Documents\work\Building-With-The-Claude-API\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Sample document

Written to match the structure the lesson's `report.md` implies. It's fictional; this version replaces the shorter `report.md` from the chunking notebook, so later lessons (BM25, multi-index) can use it too.

In [2]:
REPORT = r'''# Annual Interdisciplinary Research Review 2023

## Executive Summary
This review summarizes the year's work across five research and operations divisions. Highlights include a successful Phase II clinical trial, a major platform release from software engineering, improved financial performance, and the resolution of a significant security incident. Each section below covers goals, results, and next steps.

## Table of Contents
1. Methodology
2. Section 1: Medical Research
3. Section 2: Software Engineering
4. Section 3: Financial Analysis
5. Section 4: Cybersecurity
6. Section 5: Legal and Compliance
7. Conclusion

## Methodology
Each division reported quarterly metrics using a shared template. Teams documented their processes, tools, and outcomes, and an independent review panel validated the figures. Engineering-heavy divisions also submitted system logs and release notes, which were cross-checked against project tracking data.

## Section 1: Medical Research
The medical research team completed Phase II trials for compound NR-204, a treatment for chronic inflammation. Enrollment paused briefly after a stomach bug spread through one trial site; the bug was traced to a catering supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers, and Phase III planning has begun.

## Section 2: Software Engineering
The software engineering department shipped four major releases of the analytics platform. Engineers fixed 1,247 bugs, a 22 percent increase over the prior year, and reduced average resolution time from nine days to four. The team migrated core services to a new container platform and raised automated test coverage to 81 percent.

## Section 3: Financial Analysis
Total revenue reached 96 million dollars, up 18 percent. Operating margin improved to 14 percent on lower infrastructure costs. Research spending totaled 21 million dollars, most of it in medical research. Cash reserves at year end were 38 million dollars.

## Section 4: Cybersecurity
The security team handled 37 incidents during the year, most of them low severity. The most significant, incident INC-2023-Q4-011, involved unauthorized access to a staging database through a misconfigured API gateway. The team contained it within six hours, rotated all affected credentials, and found no evidence that customer data left the environment. A post-incident review led to mandatory configuration audits for all gateways.

## Section 5: Legal and Compliance
The legal team renewed data processing agreements with all major vendors and completed an external privacy audit with no material findings. Following incident INC-2023-Q4-011, counsel confirmed that no regulatory notification was required.

## Conclusion
The organization ends the year in a strong position. Priorities for next year include Phase III trials, expanding the analytics platform into healthcare, and continued investment in security controls.
'''

Path("report.md").write_text(REPORT, encoding="utf-8")
print(f"Wrote report.md: {len(REPORT)} characters")

Wrote report.md: 2956 characters


## 3. The pre-built helpers
### `chunk_by_section`
Splits on `## ` headers and keeps each header with its section.

In [3]:
def chunk_by_section(document_text):
    return [c.strip() for c in re.split(r"\n(?=## )", document_text) if c.strip()]

### `generate_embedding` (string or list)

One function for both cases: pass a string, get one embedding back; pass a list, get a list of embeddings from a single API call.

In [4]:
def generate_embedding(chunks, model=EMBED_MODEL, input_type="query"):
    is_list = isinstance(chunks, list)
    texts = chunks if is_list else [chunks]
    result = voyage_client.embed(texts, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

### `VectorIndex`

A minimal in-memory vector database. Each entry is an embedding plus a metadata dict holding the original text, because a search result that only returned numbers would be useless.

`search` returns `(metadata, distance)` pairs sorted by **cosine distance** = 1 − cosine similarity. Lower distance means more similar. Real systems use a vector database (pgvector, Pinecone, Chroma, etc.) for the same job at scale.

In [5]:
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.documents = []

    def add_vector(self, vector, document):
        self.vectors.append(vector)
        self.documents.append(document)

    def __len__(self):
        return len(self.vectors)

    @staticmethod
    def _cosine_distance(a, b):
        dot = sum(x * y for x, y in zip(a, b))
        norm = sqrt(sum(x * x for x in a)) * sqrt(sum(y * y for y in b))
        return 1 - dot / norm if norm else 1.0

    def search(self, query_vector, k=3):
        scored = [
            (doc, self._cosine_distance(query_vector, vec))
            for vec, doc in zip(self.vectors, self.documents)
        ]
        scored.sort(key=lambda pair: pair[1])
        return scored[:k]

## 4. The five steps
### Step 1: Chunk the text

In [6]:
with open("./report.md", "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(len(chunks), "chunks")
chunks[2]  # the table of contents

10 chunks


'## Table of Contents\n1. Methodology\n2. Section 1: Medical Research\n3. Section 2: Software Engineering\n4. Section 3: Financial Analysis\n5. Section 4: Cybersecurity\n6. Section 5: Legal and Compliance\n7. Conclusion'

### Step 2: Generate embeddings

All chunks in one call. The lesson calls `generate_embedding(chunks)` with the default `input_type`; passing `"document"` for stored chunks is Voyage's recommended practice and usually retrieves a little better.

In [7]:
embeddings = generate_embedding(chunks, input_type="document")
first_embedding = embeddings[0]
if not isinstance(first_embedding, list):
	raise TypeError("Expected each embedding to be a list of numbers")
print(len(embeddings), "embeddings,", len(first_embedding), "dimensions each")

10 embeddings, 1024 dimensions each


### Step 3: Store in the vector index

In [8]:
store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

print(len(store), "entries in the store")

10 entries in the store


### Step 4: Embed the user's question

In [9]:
question = "What did the software engineering dept do last year?"
user_embedding = generate_embedding(question)  # input_type defaults to "query"

### Step 5: Search for the most relevant chunks

In [10]:
results = store.search(user_embedding, 2)

for doc, distance in results:
    print(f"{distance:.3f}\n{doc['content'][0:200]}\n")

0.387
## Section 2: Software Engineering
The software engineering department shipped four major releases of the analytics platform. Engineers fixed 1,247 bugs, a 22 percent increase over the prior year, and

0.472
## Executive Summary
This review summarizes the year's work across five research and operations divisions. Highlights include a successful Phase II clinical trial, a major platform release from softwa



Section 2 should come first. Your distances won't match the lesson's (about 0.71 and 0.72) exactly, because the document differs, but the pattern should: the right section wins, and the runner-up (often Methodology, which mentions engineering) isn't far behind. That narrow gap is the weakness the next lessons address.

## 5. Try more questions

A helper that runs steps 4 and 5 together. Watch the last question: it hinges on an exact ID. Embeddings capture meaning, not exact strings, so ID lookups are hit-or-miss (it mentions the incident in both Section 4 and Section 5). That weakness is the motivation for BM25 in the next lesson.

In [11]:
def retrieve(question, k=2, show=True):
    results = store.search(generate_embedding(question), k)
    if show:
        print(f"Q: {question}")
        for doc, distance in results:
            print(f"   {distance:.3f}  {doc['content'].splitlines()[0]}")
        print()
    return results


retrieve("How much money did the company make?")
retrieve("Were there any problems with the clinical trials?")
retrieve("What happened with INC-2023-Q4-011?");

Q: How much money did the company make?
   0.428  ## Section 3: Financial Analysis
   0.567  ## Executive Summary



RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

## 6. (Optional) Generation: let Claude answer

The lesson stops at retrieval. The full RAG flow from the previous lesson ends by putting the retrieved chunks into a prompt so Claude answers from them.

In [ ]:
def answer_with_rag(question, k=2):
    results = retrieve(question, k, show=False)
    context = "\n\n".join(
        f'<chunk index="{i}">\n{doc["content"]}\n</chunk>' for i, (doc, _) in enumerate(results)
    )
    prompt = f"""Answer the user's question using only the report excerpts below.
If the excerpts don't contain the answer, say so.

<report_excerpts>
{context}
</report_excerpts>

<question>
{question}
</question>"""

    response = anthropic_client.messages.create(
        model=MODEL, max_tokens=500, messages=[{"role": "user", "content": prompt}],
    )
    print("Retrieved:", [doc["content"].splitlines()[0] for doc, _ in results])
    print()
    print(response.content[0].text)


answer_with_rag("What did the software engineering dept do last year?")

## Takeaways
- RAG retrieval = chunk → embed → store (embedding + original text) → embed the question → nearest-neighbor search.
- Cosine **distance** is 1 − similarity: lower is better.
- Semantic search handles paraphrased questions well but struggles with exact identifiers, which BM25 covers next.

In [12]:
count = anthropic_client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": "How many tokens is this?"}],
)
print(count.input_tokens)

13
